In [61]:
import mpqp
from mpqp import QCircuit
from mpqp.gates import H, CNOT, CustomGate, CustomControlledGate
from mpqp.measures import BasisMeasure
from mpqp.execution import run
from mpqp.execution.devices import IBMDevice
import numpy as np

In [62]:
print(mpqp.__version__)

0.6.0


In [63]:
circuit = QCircuit(
    [H(0), CNOT(0, 1), BasisMeasure([0, 1], shots=(2**5))],
    nb_qubits=2,
    label="Phi+"   
)
print(circuit)

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 


In [64]:
result = run(circuit, IBMDevice.AER_SIMULATOR)
print(f"Counts:        {result.counts}")
print(f"Probabilities: {result.probabilities}")


Counts:        [14, 0, 0, 18]
Probabilities: [0.4375 0.     0.     0.5625]


In [65]:
root2 = np.sqrt(2)

M = np.array([
    [1/root2, 1],
    [1/root2, 0]
])

ψ = np.array([1, 0])
φ = np.array([1/root2, 1/root2])

ε = np.array([
    [0, 1],
    [-1, 0]
], dtype=complex)

V = np.array([
    [φ[0], -np.conj(φ[1])],
    [φ[1],  np.conj(φ[0])],
])
V_dag = V.conj().T

U = ε @ V_dag

In [66]:
initial_state = np.kron(ψ, np.array([1, 0]))

qc = QCircuit.initializer(initial_state)

U_gate = CustomGate(U, [0], "ε @ V_dag")
cntrl_U = CustomControlledGate([1], U_gate, label="CU")

qc.add([H(1), cntrl_U, H(1)])
qc.add(BasisMeasure([1], shots=(2**16)))

print(qc)

          ┌─────────┐        
q_0: ─────┤ Unitary ├────────
     ┌───┐└────┬────┘┌───┐┌─┐
q_1: ┤ H ├─────■─────┤ H ├┤M├
     └───┘           └───┘└╥┘
c: 1/══════════════════════╩═
                           0 


In [67]:
result = run(qc, IBMDevice.AER_SIMULATOR)
print(f"Counts:        {result.counts}")
print(f"Probabilities: {result.probabilities}")

Counts:        [9533, 56003]
Probabilities: [0.14546204 0.85453796]


In [68]:
p0 = result.probabilities[0]
2 * p0 - 1

np.float64(-0.709075927734375)